# Domain anylsis with InterProScan
https://github.com/ebi-pf-team/interproscan/releases

In [ ]:
# install InterProScan, download latest verion from https://github.com/ebi-pf-team/interproscan/releases
# entpacken in the right folder
cd blasting
cd interproscan
cd interproscan-5.74-105.0
tar -xzvf interproscan-5.74-105.0-64-bit.tar.gz
cd interproscan-5.74-105.0


In [ ]:
#Test 
!./interproscan.sh -i test_all_appl.fasta -f tsv

# Create a target directory
mkdir -p ~/CODING/interproscan-5.74-105.0/results



/bin/bash: line 1: ./interproscan.sh: No such file or directory


#### Analyse mit eigener Fasta

In [2]:
# Setting file directories
input_omicron_fasta = "../data/queryOmicron.fasta"

output_file = "../results/interproscan.tsv"
output_dir = "../results"
interproscan_dir = "../interproscan-5.74-105.0"




In [17]:
# Creating tsv with interproscan: scanning protein sequence against a collection of protein signature databases
import subprocess

cmd = [
    f"{interproscan_dir}/interproscan.sh",
    "-i", input_omicron_fasta,
    "-f", "tsv",
    "-goterms",
    "-iprlookup",
    "-cpu", "4",
    "-o", output_file
]

# Ausführen
result = subprocess.run(cmd, capture_output=True, text=True)

# Status & Logs anzeigen
print("Return code:", result.returncode)
print("stdout:\n", result.stdout)
print("stderr:\n", result.stderr)

KeyboardInterrupt: 

In [4]:
# Auswertung durch pandas
import pandas as pd

# Datei einlesen
df = pd.read_csv(output_file, sep="\t", header=None)
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,7CAB_1|Chains,510eb9f9b31257fc1b9117f0cfb6e5c2,1208,Pfam,PF16451,"Betacoronavirus-like spike glycoprotein S1, N-...",33,337,1.7E-66,T,18-06-2025,IPR032500,"Spike glycoprotein S1, N-terminal domain, beta...",-,-
1,7CAB_1|Chains,510eb9f9b31257fc1b9117f0cfb6e5c2,1208,Pfam,PF09408,"Betacoronavirus spike glycoprotein S1, recepto...",349,526,1.1E-37,T,18-06-2025,IPR018548,"Spike (S) protein S1 subunit, receptor-binding...",-,-
2,7CAB_1|Chains,510eb9f9b31257fc1b9117f0cfb6e5c2,1208,ProSiteProfiles,PS51922,Betacoronavirus spike (S) glycoprotein S1 subu...,9,303,49.265945,T,18-06-2025,IPR032500,"Spike glycoprotein S1, N-terminal domain, beta...",-,-
3,7CAB_1|Chains,510eb9f9b31257fc1b9117f0cfb6e5c2,1208,ProSiteProfiles,PS51921,Betacoronavirus spike (S) glycoprotein S1 subu...,334,527,64.510162,T,18-06-2025,IPR018548,"Spike (S) protein S1 subunit, receptor-binding...",-,-
4,7CAB_1|Chains,510eb9f9b31257fc1b9117f0cfb6e5c2,1208,Pfam,PF19209,"Coronavirus spike glycoprotein S1, C-terminal",536,592,4.5E-4,T,18-06-2025,IPR043607,"Coronavirus spike glycoprotein S1, C-terminal",-,-


In [ ]:
# Add fitting column names

# TODO update columns right names!!!!!!
df.columns = [
    "Protein Accession",        # z. B. 7CAB_1
    "Sequence Identifier",      # z. B. Chains
    "Sequence MD5",             # z. B. 510eb9f9...
    "Sequence Length",          # z. B. 1208
    "Analysis",                 # z. B. Pfam, Gene3D, etc.
    "Signature Accession",      # z. B. PF16451
    "Signature Description",    # z. B. Betacoronavirus spike...
    "Start",                    # Startposition
    "End",                      # Endposition
    "Score",                    # z. B. E-value, 0.0, etc.
    "Status",                   # T (true hit)
    "Date",                     # Ausführungsdatum
    "InterPro Accession",       # z. B. IPR032500
    "InterPro Description",     # z. B. Spike glycoprotein S1...
    "GO Terms"                  # Kann leer sein oder GO:XXXX
]


df.head()

,Protein Accession,Sequence Identifier,Sequence MD5,Sequence Length,Analysis,Signature Accession,Signature Description,Start,End,Score,Status,Date,InterPro Accession,InterPro Description,GO Terms
0,7CAB_1|Chains,510eb9f9b31257fc1b9117f0cfb6e5c2,1208,Pfam,PF16451,"Betacoronavirus-like spike glycoprotein S1, N-...",33,337,1.7E-66,T,18-06-2025,IPR032500,"Spike glycoprotein S1, N-terminal domain, beta...",-,-
1,7CAB_1|Chains,510eb9f9b31257fc1b9117f0cfb6e5c2,1208,Pfam,PF09408,"Betacoronavirus spike glycoprotein S1, recepto...",349,526,1.1E-37,T,18-06-2025,IPR018548,"Spike (S) protein S1 subunit, receptor-binding...",-,-
2,7CAB_1|Chains,510eb9f9b31257fc1b9117f0cfb6e5c2,1208,ProSiteProfiles,PS51922,Betacoronavirus spike (S) glycoprotein S1 subu...,9,303,49.265945,T,18-06-2025,IPR032500,"Spike glycoprotein S1, N-terminal domain, beta...",-,-
3,7CAB_1|Chains,510eb9f9b31257fc1b9117f0cfb6e5c2,1208,ProSiteProfiles,PS51921,Betacoronavirus spike (S) glycoprotein S1 subu...,334,527,64.510162,T,18-06-2025,IPR018548,"Spike (S) protein S1 subunit, receptor-binding...",-,-
4,7CAB_1|Chains,510eb9f9b31257fc1b9117f0cfb6e5c2,1208,Pfam,PF19209,"Coronavirus spike glycoprotein S1, C-terminal",536,592,4.5E-4,T,18-06-2025,IPR043607,"Coronavirus spike glycoprotein S1, C-terminal",-,-


#### Creating a list of domänes per protein

In [ ]:
# extrahieren relevanter domain data
domain_df = df[[
    "Protein Accession", "Sequence Length", "Analysis", "Signature Accession",
    "Signature Description", "Start", "End", "Score", "InterPro Accession"
]]
domain_df.head()


,Protein Accession,Sequence Length,Analysis,Signature Accession,Signature Description,Start,End,Score,InterPro Accession
0,7CAB_1|Chains,Pfam,PF16451,"Betacoronavirus-like spike glycoprotein S1, N-...",33,337,1.7E-66,T,"Spike glycoprotein S1, N-terminal domain, beta..."
1,7CAB_1|Chains,Pfam,PF09408,"Betacoronavirus spike glycoprotein S1, recepto...",349,526,1.1E-37,T,"Spike (S) protein S1 subunit, receptor-binding..."
2,7CAB_1|Chains,ProSiteProfiles,PS51922,Betacoronavirus spike (S) glycoprotein S1 subu...,9,303,49.265945,T,"Spike glycoprotein S1, N-terminal domain, beta..."
3,7CAB_1|Chains,ProSiteProfiles,PS51921,Betacoronavirus spike (S) glycoprotein S1 subu...,334,527,64.510162,T,"Spike (S) protein S1 subunit, receptor-binding..."
4,7CAB_1|Chains,Pfam,PF19209,"Coronavirus spike glycoprotein S1, C-terminal",536,592,4.5E-4,T,"Coronavirus spike glycoprotein S1, C-terminal"


In [16]:
# databank filtered 
domain_df = domain_df[df["Analysis"].isin(["Pfam", "SUPERFAMILY", "Gene3D"])]
domain_df.head()


/tmp/ipykernel_1857256/2033008458.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  domain_df = domain_df[df["Analysis"].isin(["Pfam", "SUPERFAMILY", "Gene3D"])]


,Protein Accession,Sequence Length,Analysis,Signature Accession,Signature Description,Start,End,Score,InterPro Accession


#### Identify domain and create a heatmap and compair it with the Genfamilies -> new study information...